In [1]:
import jax.numpy as jnp
import jax
import jax.random as jrandom

from diffrax.STLA.tree import VirtualSTLATree
from diffrax import VirtualBrownianTree
import diffrax
import scipy.stats as stats
from jax import config
# config.update("jax_debug_nans", True)

In [10]:
key = jrandom.PRNGKey(5678)
bm_key, sample_key, permute_key = jrandom.split(key, 3)

# Get >80 randomly selected points; not too close to avoid discretisation error.
t0 = 0.3
t1 = 8.7
ts = jrandom.uniform(sample_key, shape=(100,), minval=t0, maxval=t1)
sorted_ts = jnp.sort(ts)
ts = []
prev_ti = sorted_ts[0]
for ti in sorted_ts[1:]:
    if ti < prev_ti + 2**-5:
        continue
    prev_ti = ti
    ts.append(ti)
ts = jnp.stack(ts)
assert len(ts) > 10
ts = jrandom.permutation(permute_key, ts)

# Get some random paths
bm_keys = jrandom.split(bm_key, 100000)
path = jax.vmap(
    lambda k: diffrax.VirtualSTLATree(
        t0=t0, t1=t1, shape=(), tol=2**-10, key=k
    )
)(bm_keys)

# Sample some points
out = []
for ti in ts:
    vals = jax.vmap(lambda p: p.evaluate(t0, ti))(path)
    out.append((ti, vals))
out = sorted(out, key=lambda x: x[0])
t, (w_t, hh_t) = out[0]
print(f"t: {t}, w: {w_t}, H: {hh_t}")

t: 0.4732099175453186, w: [-0.25326273  0.24700585 -0.92611265 ...  0.29628858 -0.3448877
  0.03346061], H: [-0.08360992 -0.03356174 -0.08823943 ...  0.24741141 -0.19784476
  0.01763346]


In [11]:

# Test their conditional statistics
for i in range(1, len(ts)-2):
    s, (w_s, hh_s) = out[i - 1]
    r, (w_r, hh_r) = out[i]
    u, (w_u, hh_u) = out[i + 1]
    
    s = s - t0
    r = r - t0
    u = u - t0
    
    su = u - s
    sr = r - s
    ru = u - r
    d = jnp.sqrt(jnp.power(sr, 3) + jnp.power(ru, 3))
    a = (1 / (2 * su * d)) * jnp.power(sr, 7 / 2) * jnp.sqrt(ru)
    b = (1 / (2 * su * d)) * jnp.power(ru, 7 / 2) * jnp.sqrt(sr)
    c = (1.0 / (jnp.sqrt(12) * d)) * jnp.power(sr, 3 / 2) * jnp.power(ru, 3 / 2)
    
    hh_su = (1.0/su) * (u * hh_u - s * hh_s - u/2 * w_s + s/2 * w_u)
    
    w_mean = w_s + (sr/su) * (w_u - w_s) + (6*sr*ru/jnp.square(su)) * hh_su
    w_std = 2 * (a+b)/su
    normalised_w = (w_r - w_mean) / w_std
    hh_mean = (s/r) * hh_s + (jnp.power(sr, 3) /(r * jnp.square(su))) * hh_su + 0.5*w_s - s/(2*r) * w_mean
    # hh_var = (jnp.square(a) + jnp.square(c))/jnp.square(r) + jnp.square((s*(a+b))/(r*su))
    hh_var = jnp.square(c/r) + jnp.square(a/r + (s*(a+b))/(r * su))
    hh_std = jnp.sqrt(hh_var)
    normalised_hh = (hh_r - hh_mean) / hh_std
    
    h_norm_mean = jnp.mean(normalised_hh)
    h_norm_std = jnp.std(normalised_hh)
    
    # w_mean_err = jnp.mean(w_mean - w_r)
    # w_var_err = w_std - jnp.std(w_r - w_mean)
    # print(f"mean err: {w_mean_err}, var err: {w_var_err}")
    
    _, pval_w = stats.kstest(normalised_w, stats.norm.cdf)
    _, pval_hh = stats.kstest(normalised_hh, stats.norm.cdf)

    # Raise if the failure is statistically significant at 10%, subject to
    # multiple-testing correction.
    print(f"pval_W: {pval_w:.8f}, pval_H: {pval_hh:.8f}")
    # print(f"H mean: {h_norm_mean}, H std: {h_norm_std}")

pval_W: 0.38599481, pval_H: 0.82631438
pval_W: 0.15977976, pval_H: 0.09895532
pval_W: 0.67680189, pval_H: 0.39387402
pval_W: 0.10799607, pval_H: 0.06243627
pval_W: 0.28191867, pval_H: 0.20120862
pval_W: 0.12743455, pval_H: 0.18551637
pval_W: 0.92420903, pval_H: 0.88628544
pval_W: 0.72199328, pval_H: 0.84497659
pval_W: 0.17308720, pval_H: 0.23562536
pval_W: 0.59274157, pval_H: 0.49192855
pval_W: 0.89942145, pval_H: 0.93336897
pval_W: 0.52385262, pval_H: 0.29440378
pval_W: 0.17788643, pval_H: 0.21152355
pval_W: 0.73333471, pval_H: 0.66580353
pval_W: 0.49108986, pval_H: 0.45264930
pval_W: 0.54191508, pval_H: 0.53687857
pval_W: 0.05588207, pval_H: 0.02543564
pval_W: 0.20775127, pval_H: 0.19131516
pval_W: 0.40103046, pval_H: 0.39108584
pval_W: 0.69220727, pval_H: 0.65437153
pval_W: 0.30380855, pval_H: 0.28657643
pval_W: 0.67590180, pval_H: 0.73663636
pval_W: 0.98543931, pval_H: 0.95855351
pval_W: 0.04741885, pval_H: 0.06456553
pval_W: 0.76926040, pval_H: 0.83676577
pval_W: 0.37610051, pval_

In [2]:
key = jrandom.PRNGKey(200)
shape_struct = jax.ShapeDtypeStruct((10000,), jnp.float32)
tree1 = VirtualSTLATree(t0=0, t1=1, tol=2**-7, shape=shape_struct, key=key)
tree2 = VirtualBrownianTree(t0=0, t1=1, tol=2**-7, shape=shape_struct, key=key)

In [3]:
def print_correct_vars(_t0, _t1):
    h = _t1 - _t0
    print(f"true hh_var: {1/12 * h:.4f}, w_var: {h:.4f}")

In [10]:
t0, t1, t2, t3 = 0.25, 0.5, 0.625, 0.75

In [13]:
w02 = tree2.evaluate(t0,t2)
w13 = tree2.evaluate(t1,t3)
w01 = tree2.evaluate(t0,t1)
w23 = tree2.evaluate(t2,t3)
w03 = tree2.evaluate(t0,t3)

In [11]:
w02, hh02 = tree1.evaluate(t0,t2)
w13, hh13 = tree1.evaluate(t1,t3)
w01, hh01 = tree1.evaluate(t0,t1)
w23, hh23 = tree1.evaluate(t2,t3)
w03, hh03 = tree1.evaluate(t0,t3)
wtot, hhtot = tree1.evaluate(0,1)

In [9]:
for w, hh in [(w02, hh02), (w13, hh13), (w01,hh01), (w23, hh23), (wtot,hhtot)]:
    print(f"hh_mean: {jnp.mean(hh):.6f}, w_mean: {jnp.mean(w):.6f}, hh_var: {jnp.var(hh):.5f}, w_var: {jnp.var(w):.5f}")

hh_mean: 0.000279, w_mean: -0.004125, hh_var: 0.03615, w_var: 0.45023
hh_mean: -0.000093, w_mean: -0.006365, hh_var: 0.02891, w_var: 0.34762
hh_mean: 0.000579, w_mean: -0.002813, hh_var: 0.02356, w_var: 0.28607
hh_mean: -0.001566, w_mean: -0.005053, hh_var: 0.01581, w_var: 0.19266
hh_mean: -0.001843, w_mean: -0.011876, hh_var: 0.08470, w_var: 1.00171


In [38]:
print_correct_vars(t0, t2)
print(f"emp  hh_var: {jnp.var(hh02):.4f}, w_var: {jnp.var(w02):.4f}")
print_correct_vars(t1, t3)
print(f"emp  hh_var: {jnp.var(hh13):.4f}, w_var: {jnp.var(w13):.4f}")
print_correct_vars(t0, t1)
print(f"emp  hh_var: {jnp.var(hh01):.4f}, w_var: {jnp.var(w01):.4f}")
print_correct_vars(t2, t3)
print(f"emp  hh_var: {jnp.var(hh23):.4f}, w_var: {jnp.var(w23):.4f}")

true hh_var: 0.0364, w_var: 0.4370
emp  hh_var: 0.0362, w_var: 0.4507
true hh_var: 0.0289, w_var: 0.3472
emp  hh_var: 0.0290, w_var: 0.3478
true hh_var: 0.0234, w_var: 0.2805
emp  hh_var: 0.0236, w_var: 0.2861
true hh_var: 0.0159, w_var: 0.1907
emp  hh_var: 0.0158, w_var: 0.1927


In [41]:
print(jnp.cov(w02, w13))
print(jnp.cov(hh02, hh13))

[[0.45072773 0.16195504]
 [0.16195504 0.3477899 ]]
[[ 0.0362108  -0.01186689]
 [-0.01186689  0.02895591]]


In [23]:
# Chen's relation
w_diff_013 = jnp.mean(jnp.abs(w03 - (w01 + w13)))
hh_chen_01_03 = (t1-t0)*hh01 + (t3-t1)*hh13 + 0.5*((t3-t0)*w01 - (t1-t0) * (w01 + w13))
hh_diff_013 = jnp.mean(jnp.abs((t3 - t0) * hh03 - hh_chen_01_03))
w_diff_023 = jnp.mean(jnp.abs(w03 - (w02 + w23)))
print(w_diff_013)
print(hh_diff_013)
print(w_diff_023)

1.1658482e-08
1.5718944e-08
1.0407902e-08
